# Traces en vrac pour check cas limites affiliation

In [ ]:
# ========= Diagnostic des cas concernés par le fallback ==========

# Cas concernés : affiliation mandat manquante, mais groupeAbrev disponible
mask_fallback = df["affiliation_mandat_députés"].isna() & df["groupeAbrev"].notna()

fallback_cases = df.loc[
    mask_fallback,
    [
        "id_acteur",
        "nom_orateur_clean",
        "qualite_orateur",
        "groupeAbrev",
        "dateSeance_ts",
    ],
].copy()

# Reconstituer l'affiliation qui serait attribuée par le fallback
fallback_cases["affiliation_fallback"] = fallback_cases["groupeAbrev"]
fallback_cases["affiliation_fallback"] = fallback_cases["affiliation_fallback"].replace(
    recodage_affiliation
)
fallback_cases["affiliation_fallback"] = fallback_cases["affiliation_fallback"].replace(
    {"LES-REP": "LR", "UMP": "LR"}
)

# Print des infos
print("=== Cas concernés par le fallback via groupeAbrev ===")
print("Nombre d'interventions concernées :", len(fallback_cases))
print("Nombre d'id_acteur uniques :", fallback_cases["id_acteur"].nunique(dropna=True))
print(
    "Nombre d'orateurs uniques :",
    fallback_cases["nom_orateur_clean"].nunique(dropna=True),
)

print("\nListe des orateurs concernés :")
print(fallback_cases["nom_orateur_clean"].dropna().unique())

display(
    fallback_cases[
        [
            "id_acteur",
            "nom_orateur_clean",
            "groupeAbrev",
            "affiliation_fallback",
        ]
    ]
    .value_counts()
    .reset_index(name="n_interventions")
)

In [ ]:
# ========= Cas limites GOUV =============

# DANS CAS AFFILIATION DYNAMIQUE DÉPUTÉS QUI SONT GOUV

# vérification des cas sans affiliation :
print(
    "Nombre d'id_acteur uniques sans affiliation :",
    df[df["affiliation_mandat_députés"].isna()]["id_acteur"].nunique(),
)
print("\nValeur counts des id_acteur sans affiliation (top):")
print(
    df[df["affiliation_mandat_députés"].isna()][["id_acteur", "nom_orateur_clean"]]
    .value_counts()
    .head()
)

# Vérifier les cas où affiliation n'est pas nulle mais avec qualité orateur spécifique
# = membres du gouv mais qui sont députés et flaguent donc avec une affiliation députés

mask_affil_with_qualite = df["affiliation_mandat_députés"].notna() & df[
    "qualite_orateur"
].str.contains("ministre|garde des sceaux|secrétaire d'État", case=False, na=False)

print(
    "Nombre de lignes avec affiliation ET qualité gouvernementale :",
    mask_affil_with_qualite.sum(),
)
print("\nAffiliations pour ces cas :")
print(
    df.loc[mask_affil_with_qualite, "affiliation_mandat_députés"].value_counts().head()
)

print("\nExemples de ces lignes :")
print(
    df.loc[
        mask_affil_with_qualite,
        ["nom_orateur_clean", "qualite_orateur", "affiliation_mandat_députés"],
    ]
    .drop_duplicates()
    .head()
)


## Pour dif id acteur vs id orateur :
Pas un pb de notre code :

parfois bug de leur fichier : deux bloc orateurs qui s'enchainent
exemple : id_syceron 3180585 dans CRSANR5L16S2023O1N292

        <paragraphe valeur_ptsodj="3" ordinal_prise="8" id_preparation="2318149" ordre_absolu_seance="348" id_acteur="PA1567" id_mandat="PM797631" id_nomination_oe="-1" id_nomination_op="-1" code_grammaire="INTERRUPTION_1_10" code_style="NORMAL" code_parole="" sommaire="0" id_syceron="3180585" valeur="795636;0 1567;0">
          <orateurs>
            <orateur>
              <nom>M. Benjamin Lucas</nom>
              <id>795636</id>
              <qualite/>
            </orateur>
            <orateur>
              <nom>M. Jérôme Guedj</nom>
              <id>1567</id>
              <qualite/>
            </orateur>
          </orateurs>
          <texte stime="7965.54">Il fallait voter l’augmentation du Smic !</texte>

Et parfois erreur de code id juste , genre ici :
rudigoz cause
tavel gueule en interruption
rudigoz continue
la président dit a tavel de pose sons cul sur sa chaise
clouet fini par brailler -> ils passent l'id_Acteur de tavel

        <paragraphe valeur_ptsodj="2" ordinal_prise="6" id_preparation="2207403" ordre_absolu_seance="310" id_acteur="PA722292" id_mandat="PM797241" id_nomination_oe="-1" id_nomination_op="-1" code_grammaire="PAROLE_GENERIQUE" code_style="NORMAL" code_parole="PAROLE_1_2" sommaire="0" id_syceron="3024322" type_debat="PLFSS" valeur="">
          <orateurs>
            <orateur>
              <nom>M. Thomas Rudigoz</nom>
              <id>722292</id>
              <qualite/>
            </orateur>
          </orateurs>
          <texte stime="5409.51">…comme lorsque M. Tavel attaque, encore une fois, M. le ministre du travail, du plein emploi et de l’insertion. Après les différents dérapages de votre groupe, vous poursuivez dans l’outrance et la violence verbales.</texte>
        </paragraphe>
        <paragraphe valeur_ptsodj="2" ordinal_prise="6" id_preparation="2207405" ordre_absolu_seance="311" id_acteur="PA794166" id_mandat="PM796749" id_nomination_oe="-1" id_nomination_op="-1" code_grammaire="INTERRUPTION_1_10" code_style="NORMAL" code_parole="" sommaire="0" id_syceron="3024324" type_debat="PLFSS" valeur="793736;0">
          <orateurs>
            <orateur>
              <nom>M. Hadrien Clouet</nom>
              <id>793736</id>
              <qualite/>
            </orateur>
          </orateurs>
          <texte stime="5420.64">Vous n’avez qu’à nous répondre !</texte>
        </paragraphe>

In [ ]:
# ==========================================================
# CAS LIMITES GOUV :
# valeurs manquantes d'affiliation_et_gouv "encadrées" par GOUV
# ==========================================================

# Colonnes utiles pour audit
cols_audit = [
    "id_acteur",
    "nom_orateur_clean",
    "dateSeance_ts",
    "affiliation_et_gouv",
    "affiliation",
    "affiliation_mandat_députés",
    "qualite_orateur",
    "code_parole",
    "id_syceron",
    "texte",
]

tmp = df.loc[
    df["id_acteur"].notna() & (df["id_acteur"] != "PA0"),
    cols_audit,
].copy()

tmp = tmp.sort_values(["id_acteur", "dateSeance_ts"]).reset_index(drop=True)
g = tmp.groupby("id_acteur", group_keys=False)

# Valeurs non manquantes les plus proches avant/après
# contre inuitif les ffill et bfill mais c'est ça
tmp["prev_non_na_affil"] = g["affiliation_et_gouv"].ffill()
tmp["next_non_na_affil"] = g["affiliation_et_gouv"].bfill()

# Dates de référence (où affiliation_et_gouv est connue)
tmp["date_affil_connue"] = tmp["dateSeance_ts"].where(
    tmp["affiliation_et_gouv"].notna()
)
tmp["date_connue_avant"] = g["date_affil_connue"].ffill()
tmp["date_connue_apres"] = g["date_affil_connue"].bfill()

# NA strictement entre deux observations GOUV
mask_na_entre_gouv = (
    tmp["affiliation_et_gouv"].isna()
    & (tmp["prev_non_na_affil"] == "GOUV")
    & (tmp["next_non_na_affil"] == "GOUV")
)

cas_na_entre_gouv = tmp.loc[
    mask_na_entre_gouv,
    [
        "id_acteur",
        "nom_orateur_clean",
        "dateSeance_ts",
        "date_connue_avant",
        "date_connue_apres",
        "qualite_orateur",
        "code_parole",
        "id_syceron",
        "affiliation_et_gouv",
        "affiliation",
        "affiliation_mandat_députés",
        "texte",
    ],
].copy()

print("Interventions NA entre deux bornes GOUV :", len(cas_na_entre_gouv))
print("id_acteur uniques concernés :", cas_na_entre_gouv["id_acteur"].nunique())
display(cas_na_entre_gouv.head(20))

# Exports pour revue manuelle
cas_na_entre_gouv.to_csv("../data/temp/cas_na_entre_gouv_lignes.csv", index=False)

# # --- Regrouper en "segments" de NA consécutifs par acteur ---
# tmp["is_gap_gouv"] = mask_na_entre_gouv
# tmp["gap_start"] = tmp["is_gap_gouv"] & ~g["is_gap_gouv"].shift(fill_value=False)
# tmp["gap_num"] = g["gap_start"].cumsum()
# tmp.loc[~tmp["is_gap_gouv"], "gap_num"] = pd.NA

# segments_na_entre_gouv = (
#     tmp.loc[tmp["is_gap_gouv"]]
#     .groupby(["id_acteur", "nom_orateur_clean", "gap_num"], dropna=False)
#     .agg(
#         date_debut_na=("dateSeance_ts", "min"),
#         date_fin_na=("dateSeance_ts", "max"),
#         nb_interventions_na=("dateSeance_ts", "size"),
#         borne_gouv_avant=("date_connue_avant", "first"),
#         borne_gouv_apres=("date_connue_apres", "first"),
#         nb_qualite_gouv_explicite=(
#             "qualite_orateur",
#             lambda s: s.astype(str)
#             .str.contains(
#                 "ministre|garde des sceaux|secrétaire d[’']État",
#                 case=False,
#                 na=False,
#                 regex=True,
#             )
#             .sum(),
#         ),
#     )
#     .reset_index()
#     .sort_values(["nb_interventions_na", "date_debut_na"], ascending=[False, True])
# )

# print("Segments NA entre bornes GOUV :", len(segments_na_entre_gouv))
# display(segments_na_entre_gouv.head(30))

# Exports pour revue manuelle
# segments_na_entre_gouv.to_csv(
#     "../data/temp/cas_na_entre_gouv_segments.csv", index=False
# )

# TODO : à partir de là, revue manuelle pour voir si on peut recoder certains NA en GOUV
# -> ok sont de la meme date visiblement, vérifier si pas de souci code
# -> check les syceron dans fichier source
# TODO : créer un code automatique si les affil avant après sont du meme jour (normalize dt)

In [ ]:
# ==========================================================
# CAS LIMITES GOUV :
# valeurs manquantes d'affiliation_et_gouv "encadrées" par GOUV
# ==========================================================

# Colonnes utiles pour audit
cols_audit = [
    "id_acteur",
    "nom_orateur_clean",
    "dateSeance_ts",
    "affiliation_et_gouv",
    "affiliation",
    "affiliation_mandat_députés",
    "qualite_orateur",
    "code_parole",
    "id_syceron",
    "texte",
]

tmp = df.loc[
    df["id_acteur"].notna() & (df["id_acteur"] != "PA0"),
    cols_audit,
].copy()

tmp = tmp.sort_values(["id_acteur", "dateSeance_ts"]).reset_index(drop=True)
g = tmp.groupby("id_acteur", group_keys=False)

# Valeurs non manquantes les plus proches avant/après
# contre inuitif les ffill et bfill mais c'est ça
tmp["prev_non_na_affil"] = g["affiliation_et_gouv"].ffill()
tmp["next_non_na_affil"] = g["affiliation_et_gouv"].bfill()

# Dates de référence (où affiliation_et_gouv est connue)
tmp["date_affil_connue"] = tmp["dateSeance_ts"].where(
    tmp["affiliation_et_gouv"].notna()
)
tmp["date_connue_avant"] = g["date_affil_connue"].ffill()
tmp["date_connue_apres"] = g["date_affil_connue"].bfill()

# NA strictement entre deux observations GOUV
mask_na_entre_gouv = (
    tmp["affiliation_et_gouv"].isna()
    & (tmp["prev_non_na_affil"] == "GOUV")
    & (tmp["next_non_na_affil"] == "GOUV")
)

cas_na_entre_gouv = tmp.loc[
    mask_na_entre_gouv,
    [
        "id_acteur",
        "nom_orateur_clean",
        "dateSeance_ts",
        "date_connue_avant",
        "date_connue_apres",
        "qualite_orateur",
        "code_parole",
        "id_syceron",
        "affiliation_et_gouv",
        "affiliation",
        "affiliation_mandat_députés",
        "texte",
    ],
].copy()

print("Interventions NA entre deux bornes GOUV :", len(cas_na_entre_gouv))
print("id_acteur uniques concernés :", cas_na_entre_gouv["id_acteur"].nunique())
display(cas_na_entre_gouv.head(20))

# Exports pour revue manuelle
cas_na_entre_gouv.to_csv("../data/temp/cas_na_entre_gouv_lignes.csv", index=False)

# ==========================================================
# AUTO-RECODAGE : NA entre GOUV si même jour que la borne
# ==========================================================

date_seance = pd.to_datetime(cas_na_entre_gouv["dateSeance_ts"]).dt.normalize()

# Masque : même jour (avec dt.normalize() que la borne avant OU après
mask_meme_jour = cas_na_entre_gouv["dateSeance_ts"].dt.normalize().eq(
    cas_na_entre_gouv["date_connue_avant"].dt.normalize()
) | cas_na_entre_gouv["dateSeance_ts"].dt.normalize().eq(
    cas_na_entre_gouv["date_connue_apres"].dt.normalize()
)

a_recoder = cas_na_entre_gouv.loc[mask_meme_jour, "id_syceron"]
a_laisser = cas_na_entre_gouv.loc[~mask_meme_jour, "id_syceron"]

print(f"Lignes à recoder automatiquement (même jour) : {mask_meme_jour.sum()}")
print(f"Lignes restant à réviser manuellement         : {(~mask_meme_jour).sum()}")

# Recodage dans df principal via id_syceron
mask_df = df["id_syceron"].isin(a_recoder) & df["affiliation_et_gouv"].isna()
df.loc[mask_df, "affiliation_et_gouv"] = "GOUV"

print(f"\nLignes recodées dans df : {mask_df.sum()}")

# Export des cas restants pour revue manuelle
cas_a_reviser = cas_na_entre_gouv.loc[~mask_meme_jour].drop(
    columns=["date_seance_norm", "date_avant_norm", "date_apres_norm"]
)
cas_a_reviser.to_csv("../data/temp/cas_na_entre_gouv_a_reviser.csv", index=False)


In [ ]:
id_pb = (
    df.loc[
        df["id_acteur"].notna()
        & df["id_orateur"].notna()
        & (df["id_acteur"] != "PA0")
        & (df["id_orateur"] != "PA0")
        & (df["id_acteur"] != df["id_orateur"])
        & (df["nom_orateur"].apply(nettoyer_nom) != df["nom_orateur_clean"]),
        [
            "id_acteur",
            "id_orateur",
            "nom_orateur",
            "nom_orateur_clean",
            "id_syceron",
        ],
    ]
    .drop_duplicates("id_syceron")
    .sort_values("id_syceron")
    .reset_index(drop=True)
)
print("\nSituations problématiques avant correction :\n")
print(id_pb.to_string(index=False))

# Corriger dans le df de base les cas listés dans id_pb,
# en remplaçant l'id_acteur erroné par le bon id_orateur.
id_pb_syceron = set(id_pb["id_syceron"].dropna().astype(str))

mask_id_pb = df["id_syceron"].astype(str).isin(id_pb_syceron)
n_lignes_corrigees = int(mask_id_pb.sum())

df.loc[mask_id_pb, "id_acteur"] = df.loc[mask_id_pb, "id_orateur"]

# Recalculer le nom recodé après correction des identifiants acteurs.
# réactualiser (puisqu'on a modifié)
most_frequent_name = df.groupby("id_acteur")["nom_orateur"].agg(
    lambda x: x.dropna().mode().iloc[0] if x.dropna().size > 0 else None
)  # version plus stable que value_counts().idxmax() en cas d'ex-aequo

# et appliquer (= la fonction change pas)
df["nom_orateur_clean"] = df.apply(get_most_frequent_name, axis=1)
df["nom_orateur_clean"] = df["nom_orateur_clean"].apply(nettoyer_nom)

id_pb_corriges_check = (
    df.loc[
        mask_id_pb,
        [
            "id_acteur",
            "id_orateur",
            "nom_orateur",
            "nom_orateur_clean",
            "id_syceron",
        ],
    ]
    .drop_duplicates("id_syceron")
    .sort_values("id_syceron")
    .reset_index(drop=True)
)

print("\nLignes corrigées dans df :", n_lignes_corrigees)
print(
    "id_syceron uniques corrigés :", int(id_pb_corriges_check["id_syceron"].nunique())
)
print("\nSituations après correction :\n")
print(id_pb_corriges_check.to_string(index=False))